# Reentrenamiento — XGBoost + LSTM con ENSO Continuo
Agente 3 (XGBoost) + Agente 4 (LSTM) con índices ENSO continuos de NOAA.
Al final calcula los pesos del ensemble igual que el sistema original.

In [ ]:
!pip install xgboost shap optuna scikit-learn -q

In [ ]:
from google.colab import files
uploaded = files.upload()   # sube dataset_features_latam_enso.csv
DATASET_PATH = list(uploaded.keys())[0]
print(f'Archivo cargado: {DATASET_PATH}')

In [ ]:
import pandas as pd
import numpy as np
import json, pickle, os
import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

SEMILLA    = 42
SEQ_LEN    = 12
OUT_DIR    = '/content/modelos_enso'
os.makedirs(OUT_DIR, exist_ok=True)

---
## Datos comunes

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f'Dataset: {df.shape}')

yearly = df.groupby(['pais', 'ano'])['casos_dengue'].transform('sum')
df = df[yearly > 100].reset_index(drop=True)
print(f'Después de filtro: {df.shape}')

max_ano   = int(df['ano'].max())
split_ano = max_ano - 2
print(f'Split: train <=>{split_ano}  |  test >{split_ano}')

---
## AGENTE 3 — XGBoost

In [ ]:
# Excluimos binarios — usamos índices continuos
COLS_EXCLUIR = [
    'iso_a0', 'pais', 'adm_1_name', 'ano', 'mes',
    'casos_dengue', 'poblacion', 'incidencia_dengue',
    'indicador_nino', 'indicador_nina',
]
COLS_FEAT = [c for c in df.columns if c not in COLS_EXCLUIR]
enso_cols = [c for c in COLS_FEAT if 'nino' in c]
print(f'Features XGBoost: {len(COLS_FEAT)}')
print(f'Columnas ENSO nuevas: {enso_cols}')

df_train = df[df['ano'] <= split_ano].copy()
df_test  = df[df['ano'] >  split_ano].copy()

X_train_raw = df_train[COLS_FEAT]
y_train     = df_train['incidencia_dengue']
X_test_raw  = df_test[COLS_FEAT]
y_test      = df_test['incidencia_dengue']
y_train_log = np.log1p(y_train)
print(f'Train: {len(df_train)}  |  Test: {len(df_test)}')

In [ ]:
# Baseline XGBoost
pipeline_base_xgb = Pipeline([
    ('imputador', SimpleImputer(strategy='median')),
    ('escalador', StandardScaler()),
    ('modelo',    XGBRegressor(random_state=SEMILLA, n_jobs=-1, verbosity=0))
])
pipeline_base_xgb.fit(X_train_raw, y_train_log)
pred_base_xgb = np.expm1(pipeline_base_xgb.predict(X_test_raw))
r2_base_xgb   = r2_score(np.log1p(y_test), np.log1p(pred_base_xgb))
mae_base_xgb  = mean_absolute_error(y_test, pred_base_xgb)
print(f'XGB Baseline — R²(log): {r2_base_xgb*100:.2f}%  |  MAE: {mae_base_xgb:.4f}')

In [ ]:
# Optuna XGBoost — K=5 folds cronológicos (últimos 5 años)
anos_train_arr  = df_train['ano'].values
fold_val_anos   = sorted(set(anos_train_arr.tolist()))[-5:]
folds_xgb = [(anos_train_arr < y, anos_train_arr == y) for y in fold_val_anos]
print(f'K=5 folds val: {fold_val_anos}')

X_tr_np     = X_train_raw.values
y_tr_log_np = y_train_log.values

def objective_xgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 1200),
        'learning_rate':    trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma':            trial.suggest_float('gamma', 0.0, 0.5),
        'objective': 'reg:squarederror',
        'random_state': SEMILLA, 'verbosity': 0, 'n_jobs': -1,
    }
    r2s = []
    for tr_mask, val_mask in folds_xgb:
        if val_mask.sum() == 0:
            continue
        imp = SimpleImputer(strategy='median')
        sc  = StandardScaler()
        Xtr = sc.fit_transform(imp.fit_transform(X_tr_np[tr_mask]))
        Xvl = sc.transform(imp.transform(X_tr_np[val_mask]))
        m   = XGBRegressor(**params)
        m.fit(Xtr, y_tr_log_np[tr_mask])
        r2s.append(r2_score(y_tr_log_np[val_mask], m.predict(Xvl)))
    return float(np.mean(r2s))

def cb_xgb(study, trial):
    best = ' <-- mejor' if trial.value == study.best_value else ''
    p = trial.params
    print(f"  Trial {trial.number+1:02d}  n_est={p['n_estimators']}  lr={p['learning_rate']:.4f}  depth={p['max_depth']}  R2_CV={trial.value*100:.2f}%{best}")

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEMILLA))
study_xgb.optimize(objective_xgb, n_trials=50, callbacks=[cb_xgb])
print(f'\nMejor R² CV XGB: {study_xgb.best_value*100:.2f}%')

In [ ]:
# Entrenar XGBoost final
best_xgb = {**study_xgb.best_params, 'objective': 'reg:squarederror',
            'random_state': SEMILLA, 'verbosity': 0, 'n_jobs': -1}
pipeline_xgb = Pipeline([
    ('imputador', SimpleImputer(strategy='median')),
    ('escalador', StandardScaler()),
    ('modelo',    XGBRegressor(**best_xgb))
])
pipeline_xgb.fit(X_train_raw, y_train_log)

pred_log_xgb = pipeline_xgb.predict(X_test_raw)
pred_xgb     = np.expm1(pred_log_xgb)
r2_xgb       = r2_score(np.log1p(y_test), pred_log_xgb)
mae_xgb      = mean_absolute_error(y_test, pred_xgb)

print(f'XGBoost final — R²(log): {r2_xgb*100:.2f}%  (antes: {r2_base_xgb*100:.2f}%  | Δ {(r2_xgb-r2_base_xgb)*100:+.2f}pp)')
print(f'               MAE:      {mae_xgb:.4f}       (antes: {mae_base_xgb:.4f}       | Δ {mae_xgb-mae_base_xgb:+.4f})')

# Lookup de predicciones XGBoost para el ensemble
xgb_ids = list(zip(df_test['iso_a0'].str.strip().str.upper(),
                   df_test['adm_1_name'].str.strip().str.upper(),
                   df_test['ano'].astype(int),
                   df_test['mes'].astype(int)))
xgb_lookup = {sid: float(p) for sid, p in zip(xgb_ids, pred_xgb)}

---
## AGENTE 4 — LSTM

In [ ]:
class DengueLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

def build_sequences(df_sorted, features, seq_len):
    X_seqs, y_vals, anos, ids = [], [], [], []
    for _, grp in df_sorted.groupby(['iso_a0', 'adm_1_name']):
        grp = grp.sort_values(['ano', 'mes'])
        if len(grp) < seq_len + 1:
            continue
        feat = grp[features].values.astype(np.float32)
        tgt  = grp['incidencia_dengue'].values.astype(np.float32)
        yr   = grp['ano'].values
        ms   = grp['mes'].values
        iso  = str(grp['iso_a0'].iloc[0]).strip().upper()
        adm  = str(grp['adm_1_name'].iloc[0]).strip().upper()
        for i in range(seq_len, len(grp)):
            X_seqs.append(feat[i - seq_len:i])
            y_vals.append(tgt[i])
            anos.append(yr[i])
            ids.append((iso, adm, int(yr[i]), int(ms[i])))
    return np.array(X_seqs), np.array(y_vals), np.array(anos), ids

# Features LSTM: 6 originales + nino34_index
LSTM_FEATS = ['tmax_promedio', 'tmin_promedio', 'precipitacion',
              'humedad_promedio', 'agua_basica', 'incidencia_dengue',
              'nino34_index']
print(f'LSTM features ({len(LSTM_FEATS)}): {LSTM_FEATS}')

X_seq, y_seq, anos_seq, seq_ids = build_sequences(df, LSTM_FEATS, SEQ_LEN)
print(f'Secuencias totales: {len(X_seq)}')

train_mask = anos_seq <= split_ano
test_mask  = anos_seq >  split_ano
X_tr  = X_seq[train_mask];  y_tr  = y_seq[train_mask]
X_te  = X_seq[test_mask];   y_te  = y_seq[test_mask]
anos_tr  = anos_seq[train_mask]
test_ids = [seq_ids[i] for i, m in enumerate(test_mask) if m]
print(f'Train: {len(X_tr)}  |  Test: {len(X_te)}')

escalador_lstm = StandardScaler()
X_tr_flat = X_tr.reshape(-1, len(LSTM_FEATS))
escalador_lstm.fit(X_tr_flat)
X_tr_sc = escalador_lstm.transform(X_tr_flat).reshape(X_tr.shape)
X_te_sc = escalador_lstm.transform(X_te.reshape(-1, len(LSTM_FEATS))).reshape(X_te.shape)
y_tr_log = np.log1p(y_tr)

In [ ]:
def entrenar_lstm(X_sc, y_log, hidden_dim, lr, dropout, epochs, seed, num_layers=2):
    torch.manual_seed(seed); np.random.seed(seed)
    m    = DengueLSTMModel(X_sc.shape[2], hidden_dim, num_layers, dropout)
    opt  = optim.Adam(m.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.MSELoss()
    Xt   = torch.tensor(X_sc, dtype=torch.float32)
    yt   = torch.tensor(y_log, dtype=torch.float32).unsqueeze(1)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=256, shuffle=True)
    m.train()
    for _ in range(epochs):
        for bx, by in loader:
            opt.zero_grad(); crit(m(bx), by).backward(); opt.step()
    return m

def evaluar_lstm(m, X_sc, y_real):
    m.eval()
    with torch.no_grad():
        pred = np.expm1(m(torch.tensor(X_sc, dtype=torch.float32)).numpy().flatten())
    return r2_score(y_real, pred), mean_absolute_error(y_real, pred)

# Baseline LSTM
modelo_base_lstm = entrenar_lstm(X_tr_sc, y_tr_log, hidden_dim=32, lr=0.01,
                                  dropout=0.0, epochs=40, seed=SEMILLA, num_layers=1)
r2_base_lstm, mae_base_lstm = evaluar_lstm(modelo_base_lstm, X_te_sc, y_te)
print(f'LSTM Baseline — R²: {r2_base_lstm*100:.2f}%  |  MAE: {mae_base_lstm:.4f}')

In [ ]:
# Optuna LSTM — 20 trials x K=5 folds
fold_val_anos_lstm = sorted(set(anos_tr.tolist()))[-5:]
folds_lstm = [(anos_tr < y, anos_tr == y) for y in fold_val_anos_lstm]
print(f'K=5 folds LSTM val: {fold_val_anos_lstm}')

def objective_lstm(trial):
    hidden_dim = trial.suggest_int('hidden_dim', 64, 512, log=True)
    num_layers = trial.suggest_int('num_layers', 1, 3)
    lr         = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    dropout    = trial.suggest_float('dropout', 0.0, 0.4)
    r2s = []
    for tr_mask, val_mask in folds_lstm:
        if tr_mask.sum() == 0 or val_mask.sum() == 0:
            continue
        Xf = X_tr[tr_mask];  yf = y_tr_log[tr_mask]
        Xv = X_tr[val_mask]; yv = y_tr[val_mask]
        sc = StandardScaler()
        Xf_sc = sc.fit_transform(Xf.reshape(-1, len(LSTM_FEATS))).reshape(Xf.shape)
        Xv_sc = sc.transform(Xv.reshape(-1, len(LSTM_FEATS))).reshape(Xv.shape)
        m = entrenar_lstm(Xf_sc, yf, hidden_dim, lr, dropout, 50, SEMILLA, num_layers)
        r2_fold, _ = evaluar_lstm(m, Xv_sc, yv)
        r2s.append(r2_fold)
    if not r2s:
        return float('-inf')
    return float(np.mean(r2s))

def cb_lstm(study, trial):
    best = ' <-- mejor' if trial.value == study.best_value else ''
    p = trial.params
    print(f"  Trial {trial.number+1:02d}  hidden={p['hidden_dim']}  layers={p['num_layers']}  lr={p['lr']:.5f}  dropout={p['dropout']:.2f}  R2_CV={trial.value*100:.2f}%{best}")

study_lstm = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEMILLA))
study_lstm.optimize(objective_lstm, n_trials=20, callbacks=[cb_lstm])
print(f'\nMejor R² CV LSTM: {study_lstm.best_value*100:.2f}%')

In [ ]:
# LSTM final — early stopping (max 300 épocas, patience=15) + ReduceLROnPlateau
mp = study_lstm.best_params
ano_val_es = max(set(anos_tr.tolist()))
val_es = anos_tr == ano_val_es
fit_es = anos_tr <  ano_val_es
Xf_es = X_tr_sc[fit_es]; yf_es = y_tr_log[fit_es]
Xv_es = X_tr_sc[val_es]; yv_es = y_tr_log[val_es]

torch.manual_seed(SEMILLA); np.random.seed(SEMILLA)
modelo_lstm = DengueLSTMModel(len(LSTM_FEATS), mp['hidden_dim'], mp['num_layers'], mp['dropout'])
opt_es  = optim.Adam(modelo_lstm.parameters(), lr=mp['lr'], weight_decay=1e-4)
sched   = optim.lr_scheduler.ReduceLROnPlateau(opt_es, patience=5, factor=0.5)
loader_es = DataLoader(
    TensorDataset(torch.tensor(Xf_es, dtype=torch.float32),
                  torch.tensor(yf_es, dtype=torch.float32).unsqueeze(1)),
    batch_size=256, shuffle=True)

best_val, es_wait, best_state, best_ep = 1e9, 0, None, 0
for ep in range(300):
    modelo_lstm.train()
    for bx, by in loader_es:
        opt_es.zero_grad(); nn.MSELoss()(modelo_lstm(bx), by).backward(); opt_es.step()
    modelo_lstm.eval()
    with torch.no_grad():
        vl = float(nn.MSELoss()(
            modelo_lstm(torch.tensor(Xv_es, dtype=torch.float32)).flatten(),
            torch.tensor(yv_es, dtype=torch.float32)))
    sched.step(vl)
    if vl < best_val - 1e-5:
        best_val = vl; es_wait = 0; best_ep = ep + 1
        best_state = {k: v.clone() for k, v in modelo_lstm.state_dict().items()}
    else:
        es_wait += 1
        if es_wait >= 15:
            print(f'Early stopping en epoch {ep+1} (mejor: epoch {best_ep})')
            break
modelo_lstm.load_state_dict(best_state)

modelo_lstm.eval()
with torch.no_grad():
    pred_log_lstm = modelo_lstm(torch.tensor(X_te_sc, dtype=torch.float32)).numpy().flatten()
pred_lstm    = np.expm1(pred_log_lstm)
r2_lstm      = r2_score(np.log1p(y_te), pred_log_lstm)
mae_lstm     = mean_absolute_error(y_te, pred_lstm)

print(f'LSTM final — R²(log): {r2_lstm*100:.2f}%  (antes: {r2_base_lstm*100:.2f}%  | Δ {(r2_lstm-r2_base_lstm)*100:+.2f}pp)')
print(f'             MAE:     {mae_lstm:.4f}       (antes: {mae_base_lstm:.4f}       | Δ {mae_lstm-mae_base_lstm:+.4f})')

---
## Ensemble — pesos proporcionales al R²

In [ ]:
xgb_preds_c, lstm_preds_c, ens_y = [], [], []
for sid, lstm_p, y_val in zip(test_ids, pred_lstm, y_te):
    xgb_p = xgb_lookup.get(sid)
    if xgb_p is not None:
        xgb_preds_c.append(xgb_p)
        lstm_preds_c.append(lstm_p)
        ens_y.append(y_val)

xgb_arr  = np.array(xgb_preds_c)
lstm_arr = np.array(lstm_preds_c)
y_arr    = np.array(ens_y)

total_r2 = r2_xgb + r2_lstm
w_xgb    = r2_xgb  / total_r2
w_lstm   = r2_lstm / total_r2

ens_log  = w_xgb * np.log1p(xgb_arr) + w_lstm * np.log1p(lstm_arr)
ens_pred = np.expm1(ens_log)
r2_ens   = r2_score(np.log1p(y_arr), ens_log)
mae_ens  = mean_absolute_error(y_arr, ens_pred)

print('=' * 55)
print('           MÉTRICAS FINALES — ENSO CONTINUO')
print('=' * 55)
print(f'  XGBoost R²(log):  {r2_xgb*100:.2f}%   MAE: {mae_xgb:.4f}')
print(f'  LSTM    R²(log):  {r2_lstm*100:.2f}%   MAE: {mae_lstm:.4f}')
print(f'  Ensemble R²(log): {r2_ens*100:.2f}%   MAE: {mae_ens:.4f}')
print(f'  Peso XGBoost: {w_xgb:.3f}   Peso LSTM: {w_lstm:.3f}')
print('=' * 55)

---
## SHAP — XGBoost con mean(|SHAP|)

In [ ]:
print('Calculando SHAP (1-2 min)...')
modelo_xgb = pipeline_xgb.named_steps['modelo']
X_te_sc    = pipeline_xgb[:-1].transform(X_test_raw)
explainer  = shap.TreeExplainer(modelo_xgb)
shap_vals  = explainer.shap_values(X_te_sc)
if isinstance(shap_vals, list): shap_vals = shap_vals[0]

shap_dict = dict(sorted(
    {f: float(v) for f, v in zip(COLS_FEAT, np.abs(shap_vals).mean(axis=0))}.items(),
    key=lambda x: x[1], reverse=True
))
print('\nTop 15 SHAP:')
for i, (k, v) in enumerate(list(shap_dict.items())[:15], 1):
    marker = ' <- ENSO nuevo' if 'nino' in k else ''
    print(f'  #{i:02d}  {k:<35} {v:.4f}{marker}')

---
## Guardar artefactos

In [ ]:
# XGBoost
with open(f'{OUT_DIR}/pipeline_ml.pkl',      'wb') as f: pickle.dump(pipeline_xgb, f)
with open(f'{OUT_DIR}/cols_feat.pkl',         'wb') as f: pickle.dump(COLS_FEAT, f)
with open(f'{OUT_DIR}/shap_importance.json',  'w') as f: json.dump(shap_dict, f, indent=4)

# LSTM
torch.save(modelo_lstm.state_dict(), f'{OUT_DIR}/lstm_model.pth')
with open(f'{OUT_DIR}/escalador_lstm.pkl',   'wb') as f: pickle.dump(escalador_lstm, f)
with open(f'{OUT_DIR}/lstm_features.pkl',    'wb') as f: pickle.dump(LSTM_FEATS, f)

# Configs
lstm_config = {
    'seq_len': SEQ_LEN, 'input_dim': len(LSTM_FEATS),
    'hidden_dim': mp['hidden_dim'], 'num_layers': mp['num_layers'],
    'lr': mp['lr'], 'dropout': mp['dropout'],
    'r2': round(float(r2_lstm), 4), 'mae': round(float(mae_lstm), 4),
    'r2_baseline': round(float(r2_base_lstm), 4),
}
with open(f'{OUT_DIR}/lstm_config.json', 'w') as f: json.dump(lstm_config, f, indent=4)

metrics_out = {
    'r2_xgb': round(float(r2_xgb), 4), 'mae_xgb': round(float(mae_xgb), 4),
    'r2_lstm': round(float(r2_lstm), 4), 'mae_lstm': round(float(mae_lstm), 4),
    'r2_ensemble': round(float(r2_ens), 4), 'mae_ensemble': round(float(mae_ens), 4),
    'w_xgb': round(float(w_xgb), 4), 'w_lstm': round(float(w_lstm), 4),
    'n_features_xgb': len(COLS_FEAT), 'n_features_lstm': len(LSTM_FEATS),
    'enso_features': enso_cols,
}
with open(f'{OUT_DIR}/metrics.json', 'w') as f: json.dump(metrics_out, f, indent=4)

print('Artefactos guardados en /content/modelos_enso/')
print('Archivos a descargar si las metricas mejoran:')
for f in ['pipeline_ml.pkl','cols_feat.pkl','shap_importance.json',
          'lstm_model.pth','escalador_lstm.pkl','lstm_features.pkl',
          'lstm_config.json','metrics.json']:
    print(f'  {f}')